# PAA KX2

The [PAA KX2](https://www.peakanalysisandautomation.com/products/plate-handlers/kx2/) (also sold as KiNEDx KX2) is a 5-axis SCARA plate-handling robot from Peak Analysis and Automation. It supports:

- [Arms](../../capabilities/arms) (pick/place, Cartesian and joint movement, freedrive teaching)

The device communicates over CAN bus using the CANopen protocol (CiA-301 + CiA-402 drive profile). A PCAN/SocketCAN-compatible CAN interface is required (e.g. PEAK PCAN-USB).

| Model | PLR Name | Gripper | Notes |
|---|---|---|---|
| KX2 | `KX2` | servo | 4 motion axes (shoulder, Z, elbow, wrist) + servo gripper |

## Setup

`setup()` opens the CAN bus, discovers nodes, reads drive parameters, enables the motion axes, and homes + closes the servo gripper.

In [1]:
from pylabrobot.paa.kx2 import KX2

kx2 = KX2()
await kx2.setup()

uptime library not available, timestamps are relative to boot time and not to Epoch UTC
2026-04-14 16:37:07,444 - pylabrobot.paa.kx2.kx2_driver - INFO - canopen: connected, nodes=[1, 2, 3, 4, 6]


## Arm capabilities

The KX2 exposes an {class}`~pylabrobot.capabilities.arms.orientable_arm.OrientableArm` on `kx2.arm`, backed by {class}`~pylabrobot.paa.kx2.KX2ArmBackend`. Gripper yaw is controlled via a single `direction` float (degrees; 0° = front). For the full arm API, see [Arms](../../capabilities/arms).

The raw driver (`kx2.driver`, a `KX2Driver`) stays available for low-level access — motor commands, drive diagnostics, binary interpreter, etc.

### Gripper control

The servo gripper is force-aware. `close_gripper` moves to the target width and then verifies the plate is gripped; pass `check_plate_gripped=False` via `GripParams` to skip the verification move. Units are KX2 gripper encoder units (mm-equivalent once calibrated).

In [2]:
await kx2.arm.open_gripper(gripper_width=30)

In [3]:
from pylabrobot.paa.kx2 import KX2ArmBackend

await kx2.arm.close_gripper(
    gripper_width=0,
    backend_params=KX2ArmBackend.GripParams(check_plate_gripped=False),
)

In [4]:
await kx2.arm.is_gripper_closed()

True

Adjust the default gripping force (as a percentage of the motor's peak current) before closing on fragile labware:

In [5]:
await kx2.arm.backend.servo_gripper_set_default_gripping_force(max_force_percent=30)

### Cartesian movement

Move the arm to a Cartesian location. `direction` is the gripper yaw in degrees (Z-axis rotation only — the KX2 cannot roll or pitch). Use {class}`~pylabrobot.paa.kx2.KX2ArmBackend.CartesianMoveParams` to override velocity and acceleration.

In [6]:
await kx2.arm.request_gripper_location()

GripperLocation(location=Coordinate(x=0.0025, y=241.699, z=150.0018), rotation=Rotation(x=0, y=0, z=255.4754734040739))

In [7]:
from pylabrobot.resources import Coordinate

await kx2.arm.move_to_location(
    location=Coordinate(x=330.124, y=210.552, z=320.0441),
    direction=198,
    backend_params=KX2ArmBackend.CartesianMoveParams(vel_pct=25, accel_pct=25),
)

### Joint movement

Move in joint space using the {class}`~pylabrobot.paa.kx2.KX2Axis` enum. Rotary axes in degrees; Z (and gripper) in mm-equivalent encoder units.

```{warning}
Moving to arbitrary joint positions can cause the arm to collide with its base, gripper, or nearby equipment. Verify coordinates in a safe workspace first, and start with low velocity.
```

In [8]:
from pylabrobot.paa.kx2 import KX2Axis

position = {
    KX2Axis.SHOULDER: 0.0,
    KX2Axis.Z: 150.0,
    KX2Axis.ELBOW: 90.0,
    KX2Axis.WRIST: 0.0,
}
await kx2.arm.backend.move_to_joint_position(
    position,
    backend_params=KX2ArmBackend.JointMoveParams(vel_pct=25, accel_pct=25),
)

### Joint-space pick and place

`pick_up_at_joint_position` / `drop_at_joint_position` do the same thing as their Cartesian counterparts (move, then close/open the gripper), but the target pose is specified in joint space. Useful when you've taught a position via freedrive and want to return to it without going through inverse kinematics.

In [9]:
pick_joints = {
    KX2Axis.SHOULDER: 0.0,
    KX2Axis.Z: 150.0,
    KX2Axis.ELBOW: 90.0,
    KX2Axis.WRIST: 0.0,
}
place_joints = {
    KX2Axis.SHOULDER: 30.0,
    KX2Axis.Z: 150.0,
    KX2Axis.ELBOW: 90.0,
    KX2Axis.WRIST: 0.0,
}

await kx2.arm.backend.pick_up_at_joint_position(pick_joints, resource_width=0)
await kx2.arm.backend.drop_at_joint_position(place_joints, resource_width=30)

### Querying position

Read the current joint angles or Cartesian end-effector pose:

In [10]:
await kx2.arm.backend.request_joint_position()

{<KX2Axis.SHOULDER: 1>: 30.0002217292257,
 <KX2Axis.Z: 2>: 150.00008664814723,
 <KX2Axis.ELBOW: 3>: 90.0000385241022,
 <KX2Axis.WRIST: 4>: 0.0,
 <KX2Axis.SERVO_GRIPPER: 6>: 0.22776546532767158}

In [11]:
await kx2.arm.request_gripper_location()

GripperLocation(location=Coordinate(x=-120.829, y=209.2797, z=150.0001), rotation=Rotation(x=0, y=0, z=30.000264644559707))

### Pick and place

`pick_up_at_location` moves to the target pose and closes the gripper to `resource_width`. `drop_at_location` moves and opens the gripper. Approach and retract motions are the caller's responsibility — bracket these calls with your own pre- and post-moves.

In [ ]:
pick  = Coordinate(x=450.0, y=0.0,    z=80.0)
place = Coordinate(x=450.0, y=-200.0, z=80.0)
above = Coordinate(x=0,     y=0,      z=60.0)
yaw   = 0.0

# pick_up_at_location stores resource_width on the frontend; drop_at_location
# reuses it, so you don't pass it again.
await kx2.arm.move_to_location(pick + above, direction=yaw)
await kx2.arm.pick_up_at_location(location=pick, direction=yaw, resource_width=0)
await kx2.arm.move_to_location(pick + above, direction=yaw)

await kx2.arm.move_to_location(place + above, direction=yaw)
await kx2.arm.drop_at_location(location=place, direction=yaw)
await kx2.arm.move_to_location(place + above, direction=yaw)

### Freedrive (teaching mode)

Freedrive disables torque on the motion axes so you can push the arm by hand; the servo gripper stays powered. The KX2 frees all motion axes at once — the `free_axes` argument is accepted for API parity and ignored.

In [2]:
await kx2.arm.backend.start_freedrive_mode(free_axes=[0])

In [3]:
# Manually guide the arm to the desired pose, then read it:
taught = await kx2.arm.request_gripper_location()
print(taught)

GripperLocation(location=Coordinate(x=353.1052, y=233.0403, z=140.0082), rotation=Rotation(x=0, y=0, z=121.77582182317468))


In [4]:
await kx2.arm.backend.stop_freedrive_mode()

CanError: Motor failed to enable (axis = 1)

### Halt and fault diagnostics

`halt` issues an emergency stop on every motion axis. Driver-level methods give you estop-line state and per-axis fault codes.

In [5]:
await kx2.arm.backend.halt()

In [ ]:
await kx2.driver.get_estop_state()

In [ ]:
from pylabrobot.paa.kx2 import KX2Axis

await kx2.driver.motor_get_fault(KX2Axis.SHOULDER)

## Barcode reader

The KX2's onboard barcode reader is a separate RS-232 device, not on the CAN bus — so it's exposed as its own {class}`~pylabrobot.paa.kx2.KX2BarcodeReader` `Device`. It can be used independently of the motor stack (even with the arm e-stopped).

Find the port with `python -m serial.tools.list_ports -v`; defaults are 9600 8N1, overridable via the `baudrate` kwarg.

In [ ]:
from pylabrobot.paa.kx2 import KX2BarcodeReader

bcr = KX2BarcodeReader(port="/dev/ttyUSB1")  # adjust to your actual port
await bcr.setup()

barcode = await bcr.barcode_scanning.scan()
print(barcode)

await bcr.stop()

## Teardown

In [ ]:
await kx2.stop()